<a href="https://colab.research.google.com/github/fidlarsyn/Introduction-Machine-Learning-with-python/blob/main/BAB_3_Unsupervised__Learning_and_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Preprocessing dan Scaling Data**
Transformasi data adalah tahap kritis agar fitur-fitur yang memiliki rentang nilai berbeda tidak mendominasi perhitungan jarak pada algoritma.

# Inisialisasi Berbagai Metode Scaling
Berikut adalah inisialisasi empat transformer utama yang disediakan oleh scikit-learn

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, Normalizer

# 1. StandardScaler: Mean=0, Varians=1
scaler_standard = StandardScaler()

# 2. MinMaxScaler: Rentang 0 sampai 1
scaler_minmax = MinMaxScaler()

# 3. RobustScaler: Menggunakan median dan kuartil (tahan terhadap outlier)
scaler_robust = RobustScaler()

# 4. Normalizer: Menyamakan panjang vektor fitur menjadi 1
scaler_norm = Normalizer()

# **Transformasi Data (.fit() dan .transform())**
Proses prapemrosesan harus mengikuti urutan fit kemudian transform.

In [ ]:
# Menggunakan MinMaxScaler sebagai contoh
scaler = MinMaxScaler()
# Mempelajari parameter (min/max) hanya dari data latih
scaler.fit(X_train)

# Transformasi data latih
X_train_scaled = scaler.transform(X_train)

print("Statistik setelah scaling (X_train_scaled):")
print("Nilai minimum fitur:", X_train_scaled.min(axis=0))
print("Nilai maksimum fitur:", X_train_scaled.max(axis=0))
2.3 Konsistensi Data Latih dan Data Uji (Menghindari Data Leakage)
Salah satu kesalahan fatal dalam machine learning adalah melakukan fit ulang pada data uji. Kita harus menggunakan parameter yang sama dari data latih.
# Transformasi data uji menggunakan parameter yang dipelajari dari X_train
# PERINGATAN: Jangan gunakan scaler.fit(X_test) karena akan menyebabkan data leakage
X_test_scaled = scaler.transform(X_test)

# Statistik data uji tidak harus tepat 0-1 karena menggunakan min/max dari data latih
print("Nilai minimum data uji setelah scaling:", X_test_scaled.min(axis=0))

## **Reduksi Dimensi (Dimensionality Reduction)**

# Principal Component Analysis (PCA)
PCA merotasi dataset untuk menemukan arah varians maksimum. Di sini kita mengekstraksi atribut components_ untuk memahami komposisi fitur.

In [ ]:
from sklearn.decomposition import PCA

# Reduksi data dari 30 fitur menjadi 2 komponen utama
pca = PCA(n_components=2, random_state=42)
pca.fit(X_train_scaled)

# Transformasi data
X_pca = pca.transform(X_train_scaled)

# Menampilkan informasi komponen utama
print("Bentuk komponen utama (n_components, n_features):", pca.components_.shape)
print("Isi komponen utama:\n", pca.components_)

# Visualisasi
plt.figure(figsize=(8, 8))
mglearn.discrete_scatter(X_pca[:, 0], X_pca[:, 1], y_train)
plt.legend(cancer.target_names, loc="best")
plt.xlabel("Komponen Utama Pertama")
plt.ylabel("Komponen Utama Kedua")

# Non-Negative Matrix Factorization (NMF)
NMF berguna untuk dekomposisi fitur di mana data asli tidak memiliki nilai negatif, sering menghasilkan komponen yang lebih mudah diinterpretasikan.

In [ ]:
from sklearn.decomposition import NMF

# Inisialisasi NMF untuk mengekstrak 15 komponen
nmf = NMF(n_components=15, random_state=42)
nmf.fit(X_train_scaled)
X_nmf = nmf.transform(X_train_scaled)

print("Bentuk data setelah NMF:", X_nmf.shape)

## t-Distributed Stochastic Neighbor Embedding (t-SNE)
t-SNE adalah algoritma manifold learning yang sangat kuat untuk visualisasi.

In [ ]:
from sklearn.manifold import TSNE

digits = load_digits()
tsne = TSNE(random_state=42)

# t-SNE hanya memiliki fit_transform. Algoritma ini tidak mendukung
# transformasi data baru secara terpisah karena sifatnya yang iteratif
# dan berfokus pada hubungan antar poin data saat ini.
digits_tsne = tsne.fit_transform(digits.data)

plt.figure(figsize=(10, 10))
for i in range(len(digits.data)):
    plt.text(digits_tsne[i, 0], digits_tsne[i, 1], str(digits.target[i]),
             color=plt.cm.tab10(digits.target[i] / 10.))
plt.xlabel("t-SNE feature 0")
plt.ylabel("t-SNE feature 1")

# **Algoritma Klastering (Clustering)**



# k-Means Clustering

In [ ]:
# Mencari 3 klaster menggunakan metode K-Means
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(X_train_scaled)

print("Label klaster yang ditemukan:\n", kmeans.labels_)

# Agglomerative Clustering dan Dendrogram
Klastering hierarkis memungkinkan kita melihat hubungan antar data secara visual melalui dendrogram.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, ward

# Menggunakan 20 sampel data latih untuk visualisasi dendrogram yang bersih
X_subset = X_train_scaled[:20]
linkage_array = ward(X_subset)
dendrogram(linkage_array)

# Prediksi label menggunakan Agglomerative
agg = AgglomerativeClustering(n_clusters=3)
assignment = agg.fit_predict(X_train_scaled)

# DBSCAN
Algoritma berbasis kepadatan yang secara otomatis dapat mengidentifikasi noise.

In [ ]:
from sklearn.cluster import DBSCAN

# Parameter eps mengontrol radius pencarian tetangga
dbscan = DBSCAN(eps=0.5, min_samples=5)
clusters = dbscan.fit_predict(X_train_scaled)

print("Klaster yang terdeteksi (-1 adalah noise):\n", clusters)

# **Evaluasi dan Perbandingan Algoritma Klastering**
Evaluasi pada unsupervised learning dilakukan menggunakan dataset buatan untuk mengukur efektivitas algoritma terhadap ground truth menggunakan Adjusted Rand Index (ARI).

In [ ]:
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.datasets import make_moons

# Membuat dataset buatan moon
X_moons, y_moons = make_moons(n_samples=200, noise=0.05, random_state=42)
scaler = StandardScaler().fit(X_moons)
X_scaled = scaler.transform(X_moons)

# List algoritma untuk dibandingkan
algorithms = [KMeans(n_clusters=2), AgglomerativeClustering(n_clusters=2), DBSCAN()]

for algo in algorithms:
    labels = algo.fit_predict(X_scaled)
    # ARI digunakan untuk membandingkan hasil klastering dengan label asli (ground truth)
    ari = adjusted_rand_score(y_moons, labels)
    # Silhouette digunakan untuk evaluasi tanpa label (mengukur kepadatan klaster)
    sil = silhouette_score(X_scaled, labels)
    print(f"Algoritma: {algo.__class__.__name__}")
    print(f"ARI: {ari:.2f} | Silhouette: {sil:.2f}")